# 01 — Treinando um modelo causal “puro” do zero

Objetivo: mostrar o ciclo completo **dados → tokenizer → arquitetura aleatória → treino → geração** sem baixar pesos pré-treinados.

O modelo usa a implementação `GPT2LMHeadModel` apenas como uma arquitetura Transformer causal pronta; **todos os pesos começam aleatórios**. O tokenizer também é treinado localmente no corpus.

Para um teste rápido, o caderno cria um corpus minúsculo. Para aprender algo útil, substitua `data/corpus.txt` por muitos MB/GB de texto limpo.

In [ ]:
%pip install -U "transformers>=4.51.0" "tokenizers>=0.20" "datasets>=3" "accelerate>=1"

In [ ]:
from pathlib import Path
import random, math, os
import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DATA = Path('data/corpus.txt')
DATA.parent.mkdir(exist_ok=True)

if not DATA.exists():
    linhas = [
        'Modelos de linguagem aprendem padrões estatísticos em sequências de tokens.',
        'Um Transformer causal tenta prever o próximo token usando apenas o contexto anterior.',
        'Treinar do zero permite observar claramente a diferença entre pré-treinamento e refinamento.',
        'A qualidade depende principalmente de dados, escala, otimização e avaliação.',
        'Este corpus é propositalmente pequeno e serve apenas para validar o pipeline.',
        'Raciocínio, programação, matemática e linguagem natural exigem datasets muito maiores.',
    ]
    texto = "\n".join(linhas) + "\n"
    # Repetição somente para o smoke test ter passos suficientes.
    DATA.write_text((texto + "\n") * 300, encoding='utf-8')

print(DATA, '->', DATA.stat().st_size, 'bytes')

## 1. Treinar um tokenizer Byte-Level BPE

Em produção, um tokenizer é um projeto à parte. Aqui treinamos um BPE pequeno para tornar o pipeline autocontido.

In [ ]:
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from pathlib import Path

TOKENIZER_DIR = Path('artifacts/tokenizer_puro')
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

tok = Tokenizer(models.BPE(unk_token='<unk>'))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
trainer_bpe = trainers.BpeTrainer(
    vocab_size=2000,
    min_frequency=2,
    # Sem o alfabeto inicial, um corpus curto pode acabar com apenas os tokens especiais.
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    special_tokens=['<pad>', '<s>', '</s>', '<unk>'],
)
tok.train([str(DATA)], trainer=trainer_bpe)
# `tok.model.save` grava apenas vocab.json+merges.txt (o MODELO BPE). Para recarregar o
# tokenizer completo depois (`Tokenizer.from_file`), é preciso o `tok.save` com caminho
# de arquivo — em diretório ele falha no Windows.
tok.save(str(TOKENIZER_DIR / 'tokenizer.json'))
print('Tokenizer salvo em', TOKENIZER_DIR)


In [ ]:
from transformers import PreTrainedTokenizerFast

# No Transformers 5, GPT2TokenizerFast(vocab_file=..., merges_file=...) não
# reconstrói mais um tokenizer BPE local. Preservamos o objeto treinado em memória.
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tok,
    bos_token='<s>', eos_token='</s>', unk_token='<unk>', pad_token='<pad>',
)
print('Vocab:', len(tokenizer))
print(tokenizer.tokenize('Treinar um Transformer do zero é instrutivo.'))


## 2. Criar um Transformer pequeno com pesos aleatórios

Mude `N_LAYER`, `N_HEAD` e `N_EMBD` para demonstrar custo × capacidade. O exemplo padrão fica pequeno o suficiente para laboratório.

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

BLOCK_SIZE = 128
N_LAYER = 4
N_HEAD = 4
N_EMBD = 256

config = GPT2Config(
    vocab_size=len(tokenizer),
    n_positions=BLOCK_SIZE,
    n_ctx=BLOCK_SIZE,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_embd=N_EMBD,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)
model = GPT2LMHeadModel(config)
params = sum(p.numel() for p in model.parameters())
print(f'Parâmetros: {params/1e6:.2f} M')

## 3. Transformar o corpus em blocos de tokens

In [ ]:
from datasets import Dataset

text = DATA.read_text(encoding='utf-8')
ids = tokenizer(text, add_special_tokens=False)['input_ids']

# blocos independentes com EOS ao final
chunks = []
stride = BLOCK_SIZE - 1
for i in range(0, len(ids) - 2, stride):
    chunk = ids[i:i+stride] + [tokenizer.eos_token_id]
    if len(chunk) >= 16:
        chunks.append({'input_ids': chunk})

ds = Dataset.from_list(chunks).train_test_split(test_size=0.1, seed=SEED)
print(ds)

In [ ]:
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 4. Treinar

O objetivo causal é simples: minimizar a cross-entropy do próximo token. Para o smoke test, poucas épocas bastam; o resultado não deve ser interpretado como um LLM útil.

In [ ]:
from inspect import signature
from transformers import TrainingArguments, Trainer

OUT = 'artifacts/modelo_puro'
argumentos = dict(
    output_dir=OUT,
    num_train_epochs=3,
    per_device_train_batch_size=8 if torch.cuda.is_available() else 2,
    per_device_eval_batch_size=8 if torch.cuda.is_available() else 2,
    learning_rate=5e-4,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    report_to='none',
    seed=SEED,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
)
# `overwrite_output_dir` existia no Transformers 4, mas foi removido no 5.
if 'overwrite_output_dir' in signature(TrainingArguments).parameters:
    argumentos['overwrite_output_dir'] = True
args = TrainingArguments(**argumentos)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds['train'],
    eval_dataset=ds['test'],
    data_collator=collator,
)
trainer.train()


In [ ]:
metrics = trainer.evaluate()
loss = metrics.get('eval_loss')
print(metrics)
if loss is not None and loss < 20:
    print('Perplexidade aproximada:', math.exp(loss))

## 5. Gerar texto

In [ ]:
model.eval()
device = model.device
prompt = 'Um Transformer causal aprende'
inputs = tokenizer(prompt, return_tensors='pt').to(device)
comprimento = inputs['input_ids'].shape[-1]
if comprimento == 0:
    raise RuntimeError('O tokenizer produziu uma entrada vazia; execute novamente as células 3 e 4.')
if comprimento >= BLOCK_SIZE:
    raise RuntimeError(f'O prompt tem {comprimento} tokens, mas o contexto é {BLOCK_SIZE}.')

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=min(80, BLOCK_SIZE - comprimento),
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(out[0], skip_special_tokens=True))


In [ ]:
# Salvar o artefato final completo
trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)
print('Modelo + tokenizer:', OUT)

## Exercícios de treinamento

1. Aumente o corpus e compare `eval_loss`.
2. Mude o tamanho do tokenizer (2k → 8k) e observe tokens por sentença.
3. Dobre `N_LAYER` e meça VRAM/tempo.
4. Crie um corpus com domínio específico e compare geração antes/depois.
5. Adicione um conjunto de validação realmente separado; não use texto repetido em experimentos sérios.